In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F
df_purchases_raw = spark.table("lh_bronze_game.purchases_raw")

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 3, Finished, Available, Finished, False)

In [2]:
print("Row count:", df_purchases_raw.count())
print("Column count:", len(df_purchases_raw.columns))

df_purchases_raw.printSchema()

display(df_purchases_raw.limit(5))

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 4, Finished, Available, Finished, False)

Row count: 3525
Column count: 8
root
 |-- currency: string (nullable = true)
 |-- days_since_install: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- price_usd: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- purchase_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)



SynapseWidget(Synapse.DataFrame, f588f81f-010e-4124-b7fb-73261a024e1e)

In [3]:
df_purchases_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_purchases_raw.columns
]).show(truncate=False)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 5, Finished, Available, Finished, False)

+--------+------------------+---------+---------+------------+-----------+----------+---------+
|currency|days_since_install|player_id|price_usd|product_type|purchase_id|session_id|timestamp|
+--------+------------------+---------+---------+------------+-----------+----------+---------+
|0       |0                 |0        |0        |0           |0          |4         |0        |
+--------+------------------+---------+---------+------------+-----------+----------+---------+



In [4]:
total_purchases = df_purchases_raw.count()

unique_purchase_ids = (
    df_purchases_raw
    .select("purchase_id")
    .distinct()
    .count()
)

print("Total purchases:", total_purchases)
print("Unique purchase_id:", unique_purchase_ids)
print("Duplicate:", total_purchases - unique_purchase_ids)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 6, Finished, Available, Finished, False)

Total purchases: 3525
Unique purchase_id: 3522
Duplicate: 3


In [5]:
df_purchases_clean = df_purchases_raw.dropDuplicates()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 7, Finished, Available, Finished, False)

In [6]:
print("Raw:", df_purchases_raw.count())
print("Clean:", df_purchases_clean.count())

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 8, Finished, Available, Finished, False)

Raw: 3525
Clean: 3522


In [7]:
df_purchase_check = (
    df_purchases_clean
    .withColumn(
        "price_usd_parsed",
        F.col("price_usd").cast("double")
    )
    .withColumn(
        "timestamp_parsed",
        F.to_timestamp("timestamp")
    )
)

df_purchase_check.select(
    F.count(
        F.when(F.col("price_usd_parsed").isNull(), 1)
    ).alias("invalid_price_usd"),

    F.count(
        F.when(F.col("timestamp_parsed").isNull(), 1)
    ).alias("invalid_timestamp")
).show()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 9, Finished, Available, Finished, False)

+-----------------+-----------------+
|invalid_price_usd|invalid_timestamp|
+-----------------+-----------------+
|                0|                0|
+-----------------+-----------------+



In [8]:
df_purchases_clean = (
    df_purchases_clean
    .withColumn(
        "price_usd",
        F.col("price_usd").cast("double")
    )
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp")
    )
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 10, Finished, Available, Finished, False)

In [9]:
df_purchases_clean.printSchema()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 11, Finished, Available, Finished, False)

root
 |-- currency: string (nullable = true)
 |-- days_since_install: long (nullable = true)
 |-- player_id: string (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- product_type: string (nullable = true)
 |-- purchase_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [10]:
df_purchases_clean.select(
    F.min("price_usd").alias("min_price"),
    F.max("price_usd").alias("max_price"),
    F.min("days_since_install").alias("min_days_since_install"),
    F.max("days_since_install").alias("max_days_since_install")
).show()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 12, Finished, Available, Finished, False)

+---------+---------+----------------------+----------------------+
|min_price|max_price|min_days_since_install|max_days_since_install|
+---------+---------+----------------------+----------------------+
|      1.0|   181.97|                     0|                    86|
+---------+---------+----------------------+----------------------+



In [11]:
purchase_with_install = (
    df_purchases_clean.alias("pur")
    .join(
        spark.table("lh_silver_game.players_clean")
            .select("player_id", "install_date")
            .alias("p"),
        on="player_id",
        how="left"
    )
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 13, Finished, Available, Finished, False)

In [12]:
purchase_with_install = (
    df_purchases_clean.alias("pur")
    .join(
        spark.table("lh_silver_game.players_clean")
            .select("player_id", "install_date")
            .alias("p"),
        on="player_id",
        how="left"
    )
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 14, Finished, Available, Finished, False)

In [13]:
invalid_purchase_dates = (
    purchase_with_install
    .filter(
        F.to_date("timestamp") < F.col("install_date")
    )
    .count()
)

print("Purchase before install:", invalid_purchase_dates)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 15, Finished, Available, Finished, False)

Purchase before install: 1


In [14]:
display(
    purchase_with_install
    .filter(
        F.to_date("timestamp") < F.col("install_date")
    )
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e085f5d4-99b0-4c9f-bb39-27bd06a3d7d8)

In [15]:
df_purchases_clean = (
    purchase_with_install
    .filter(
        F.to_date("timestamp") >= F.col("install_date")
    )
    .drop("install_date")
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 17, Finished, Available, Finished, False)

In [16]:
print(
    df_purchases_clean
    .join(
        spark.table("lh_silver_game.players_clean")
            .select("player_id", "install_date"),
        on="player_id",
        how="left"
    )
    .filter(F.to_date("timestamp") < F.col("install_date"))
    .count()
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 18, Finished, Available, Finished, False)

0


In [17]:
df_purchases_clean = (
    df_purchases_clean
    .withColumn(
        "session_id",
        F.when(
            F.col("session_id").isNull(),
            F.lit("Unknown")
        ).otherwise(F.col("session_id"))
    )
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 19, Finished, Available, Finished, False)

In [18]:
df_purchases_clean.filter(
    F.col("session_id").isNull()
).count()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 20, Finished, Available, Finished, False)

0

In [19]:
orphan_purchase_players = (
    df_purchases_clean.alias("pur")
    .join(
        spark.table("lh_silver_game.players_clean").alias("p"),
        F.col("pur.player_id") == F.col("p.player_id"),
        "left_anti"
    )
)

print(
    "Players tablosunda bulunmayan purchase:",
    orphan_purchase_players.count()
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 21, Finished, Available, Finished, False)

Players tablosunda bulunmayan purchase: 0


In [20]:
orphan_purchase_sessions = (
    df_purchases_clean
    .filter(F.col("session_id") != "Unknown")
    .alias("pur")
    .join(
        spark.table("lh_silver_game.sessions_clean").alias("s"),
        F.col("pur.session_id") == F.col("s.session_id"),
        "left_anti"
    )
)

print(
    "Sessions tablosunda bulunmayan purchase:",
    orphan_purchase_sessions.count()
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 22, Finished, Available, Finished, False)

Sessions tablosunda bulunmayan purchase: 4


In [21]:
unmatched_session_ids = (
    orphan_purchase_sessions
    .select("session_id")
    .distinct()
)

df_purchases_clean = (
    df_purchases_clean.alias("p")
    .join(
        unmatched_session_ids
        .withColumn("is_unmatched", F.lit(1))
        .alias("u"),
        on="session_id",
        how="left"
    )
    .withColumn(
        "session_id",
        F.when(
            F.col("is_unmatched") == 1,
            F.lit("Unknown")
        ).otherwise(F.col("session_id"))
    )
    .drop("is_unmatched")
)

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 23, Finished, Available, Finished, False)

In [22]:
print("Final row count:", df_purchases_clean.count())

df_purchases_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in [
        "purchase_id",
        "player_id",
        "price_usd",
        "product_type",
        "currency",
        "timestamp",
        "days_since_install"
    ]
]).show()

df_purchases_clean.printSchema()

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 24, Finished, Available, Finished, False)

Final row count: 3521
+-----------+---------+---------+------------+--------+---------+------------------+
|purchase_id|player_id|price_usd|product_type|currency|timestamp|days_since_install|
+-----------+---------+---------+------------+--------+---------+------------------+
|          0|        0|        0|           0|       0|        0|                 0|
+-----------+---------+---------+------------+--------+---------+------------------+

root
 |-- session_id: string (nullable = true)
 |-- player_id: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- days_since_install: long (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- product_type: string (nullable = true)
 |-- purchase_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [23]:
df_purchases_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_silver_game.purchases_clean")

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 25, Finished, Available, Finished, False)

In [24]:
df_check = spark.table("lh_silver_game.purchases_clean")

print("Saved row count:", df_check.count())
display(df_check.limit(5))

StatementMeta(, 3648170b-0f8e-4928-a081-bd13eeec51aa, 26, Finished, Available, Finished, False)

Saved row count: 3521


SynapseWidget(Synapse.DataFrame, b8ef92ed-2276-4cd0-9748-2f09f4c29131)